In [1]:
import numpy as np
import torch
from rlaopt.atoms.affine_atom import AffineAtom
from rlaopt.atoms.l1norm import L1Norm
from rlaopt.atoms.sum_squares import SumSquares
from rlaopt.expression.bilevel_expression import BilevelExpression
from rlaopt.expression.variable import Variable
from rlaopt.solvers.configs import ProxGradConfig
from rlaopt.solvers.proximal_gradient.prox_grad import ProximalGradient

In [2]:
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

In [3]:
n, p = 1024, 128
ntst = 256
s = 32 
J = np.random.choice(p, s)

In [4]:
# Lasso problem
xStar = torch.zeros(p)
xStar[J] = torch.randn(s) / (s ** 0.5)
A = torch.randn(n, p) / (n ** 0.5)
Atst = torch.randn(ntst, p) / (ntst ** 0.5)
b = A @ xStar + 0.001 * torch.randn(n) 
btst = Atst @ xStar + 0.001 * torch.randn(ntst)

In [5]:
# Init params + reg
x = Variable(torch.zeros(p))
mu = 0.1 * torch.linalg.norm(A.T @ b, ord=torch.inf)

In [6]:
# Lasso problem
F = SumSquares(AffineAtom(x, A, -b)) + L1Norm(x, scaling=mu)

In [7]:
# Step size is reciprocal of Lipschitz constant
eta = 1 / (2 * torch.linalg.norm(A,ord=2) ** 2)

In [8]:
config = ProxGradConfig(eta = eta, tol=1e-6, use_acceleration=True, use_linesearch=False)
opt = ProximalGradient(F, config)

In [9]:
# Solve the problem using step method
params = F.params
state = opt.init_state(params)
for i in range(1000):
    params, state = opt.step(params, state)
print(state.err)

tensor(0., grad_fn=<DivBackward0>)


In [10]:
# Solve the problem using solve method
params, err = opt.solve(F)
# Print norm of gradient mapping
print(err)

tensor(5.8639e-06, grad_fn=<DivBackward0>)


In [11]:
# Bilevel Expression for optimizing lasso regularization parameter
ftr = lambda mu: SumSquares(AffineAtom(x, A, -b)) + L1Norm(x, mu)
ftst = SumSquares(AffineAtom(x, Atst, -btst))
obj = BilevelExpression(mu, ftr, ftst, config, ProximalGradient)

In [12]:
# Setup solver for the bilevel problem
config_blvl = ProxGradConfig(
    eta = torch.tensor(0.001), 
    max_iters=500,
    tol = 1e-3, 
    use_acceleration=False, 
    use_linesearch=False
)
opt_blvl = ProximalGradient(obj, config_blvl) 

In [13]:
# Initial objective value
obj.forward()

tensor(0.0112, grad_fn=<SumBackward0>)

In [14]:
# Solve the bilevel problem using solve method
mu_star, err = opt_blvl.solve(obj)

In [15]:
# Final objective value
obj.evaluate(mu_star)

tensor(0.0003, grad_fn=<SumBackward0>)

In [16]:
# Print the approximate optimal regularization parameter
mu_star.value

{'w': tensor(0.0012, grad_fn=<SubBackward0>)}

In [17]:
# Print gradient norm of the objective at mu_star
torch.func.grad(obj.evaluate)(mu_star).norm().item()

0.0002811158082996038